In [1]:
import pandas as pd

from analytics.db import query

In [2]:
query(""" 
      -- W07
     SELECT 
    order_status, 
    COUNT(*) AS order_count
    FROM orders
    WHERE order_status != 'delivered' 
    AND order_delivered_customer_date IS NOT NULL
    GROUP BY order_status
    ORDER BY order_count DESC;
      """)

,order_status,order_count
0,canceled,6


In [3]:
orders = query("SELECT * FROM orders")
cleaned = orders.dropna()

len(orders), len(cleaned)

(99441, 96461)

**Explanation:** This code loads the orders table and removes any row containing missing values using `dropna()`, then compares the row count before and after cleaning, showing a loss of 2,980 orders (about 3% of total orders).

In [4]:
pd.DataFrame({
       "before": orders["order_status"].value_counts(),
       "after": cleaned["order_status"].value_counts(),
   })

,before,after
order_status,,
approved,2,NaN
canceled,625,6.0
created,5,NaN
delivered,96478,96455.0
invoiced,314,NaN
processing,301,NaN
shipped,1107,NaN
unavailable,609,NaN


**Explanation:** This code creates a comparison dataframe of order statuses before and after using `dropna()`, clearly showing which statuses disappeared completely (such as shipped, unavailable, etc.) with none left.

In [5]:

assert len(cleaned[(cleaned["order_status"] == "canceled") & (cleaned["order_delivered_customer_date"].notna())]) == 6
print("Verification passed: Exactly 6 canceled orders with delivery dates were kept by dropna().")

Verification passed: Exactly 6 canceled orders with delivery dates were kept by dropna().


**Explanation:** This code verifies that the 6 canceled orders retained after applying `dropna()` are precisely the same 6 anomalous canceled orders identified in the SQL warm-up query.

In [6]:
canceled_before = (orders["order_status"] == "canceled").mean()
canceled_after = (cleaned["order_status"] == "canceled").mean()

round(100 * canceled_before, 2), round(100 * canceled_after, 2)

(np.float64(0.63), np.float64(0.01))

**Explanation:** This code calculates the share of canceled orders as a percentage of total orders before and after cleaning, showing how random deletion distorted the metric to 0.01% instead of the true 0.63%.

In [7]:
orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

**Explanation:** This code filters for orders that have a missing customer delivery date and counts them grouped by their order status to understand the pattern behind the missing values.

In [8]:
reviews = query("SELECT * FROM order_reviews")
marked = reviews.assign(no_message=reviews["review_comment_message"].isna())

(100 * marked.groupby("review_score")["no_message"].mean()).round(1)

review_score
1    23.5
2    31.9
3    56.5
4    68.8
5    64.1
Name: no_message, dtype: float64

**Explanation:** This code loads the reviews table, flags missing review comments, and calculates the percentage of missing messages for each review score to see how customer satisfaction correlates with leaving a message.

In [9]:
products = query("SELECT * FROM products")
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

**Explanation:** This code queries the products table and counts the total number of missing values for each column to identify data gaps in weights and dimensions.

In [10]:
translation = query("SELECT * FROM category_translation")

EXTRA_TRANSLATIONS = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"],
    "product_category_name_english": ["pc_gamer", "portable_kitchen_and_food_preparers"],
})
all_translations = pd.concat([translation, EXTRA_TRANSLATIONS], ignore_index=True)

products_en = products.merge(all_translations, on="product_category_name", how="left", validate="many_to_one")
products_en["category"] = products_en["product_category_name_english"].fillna("unknown")
(products_en["category"] == "unknown").sum()

np.int64(610)

**Explanation:** This code combines official and extra category translations, merges them with the products table, fills missing English categories with the label 'unknown', and verifies the count of unknown categories.

In [11]:
#1.a
orders[orders["order_approved_at"].isna()]["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

**Explanation:** This code counts orders missing an approval timestamp grouped by their status, showing that unapproved orders are mostly created or canceled.

In [12]:
#1.b
orders_reviews = orders.merge(reviews, on="order_id", how="left")
orders_reviews[orders_reviews["review_id"].isna()]["order_status"].value_counts()

order_status
delivered      646
shipped         75
canceled        20
unavailable     14
processing       6
invoiced         5
created          2
Name: count, dtype: int64

**Explanation:** This code merges orders with reviews to find orders lacking any review at all, grouped by order status, highlighting customer feedback gaps.

## Part 2 — Copies of rows

In [13]:
for table in ["orders", "order_items", "order_payments", "order_reviews", "products", "sellers", "customers"]:
       df = query(f"SELECT * FROM {table}")
       print(table, len(df), df.duplicated().sum())

orders 99441 0
order_items 112650 0
order_payments 103886 0
order_reviews 99224 0
products 32951 0
sellers 3095 0
customers 99441 0


**Explanation:** This code loops through the seven main tables, checks for exact row duplicates using duplicated(), and prints the total row count and duplicate count for each table.

In [14]:
sellers = query("SELECT * FROM sellers")
twice = pd.concat([sellers, sellers])

len(twice), twice.duplicated().sum(), len(twice.drop_duplicates())

(6190, np.int64(3095), 3095)

**Explanation:** This code intentionally duplicates the sellers table using pd.concat to simulate a loading error, then demonstrates how drop_duplicates() removes the redundant copies.

In [15]:
items = query("SELECT * FROM order_items")
look_alike = items.duplicated(subset=["order_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"])

look_alike.sum(), round(items[look_alike]["price"].sum(), 2)

(np.int64(10225), np.float64(847719.7))

**Explanation:** This code checks for duplicate items in order_items while excluding the unique item ID column, showing that look-alike items represent multiple units of the same product rather than data errors.

In [16]:
#2.a
items = query("SELECT * FROM order_items")
payments = query("SELECT * FROM order_payments")

items.duplicated(subset=["order_id", "order_item_id"]).sum(), payments.duplicated(subset=["order_id", "payment_sequential"]).sum()

(np.int64(0), np.int64(0))

**Explanation:** This code checks for duplicate primary keys in both the order_items and order_payments tables using subset, confirming that their primary keys have no exact duplicates.

In [17]:
#2.b
products = query("SELECT * FROM products")
product_cols = [c for c in products.columns if c != "product_id"]
products.duplicated(subset=product_cols).sum()

np.int64(695)

**Explanation:** This code checks if any products share identical attributes across all columns except for their product_id to detect duplicate listings or catalog errors.

## Part 3 — Values that cannot be true

In [18]:
items["price"].min(), items["price"].max(), (items["freight_value"] == 0).sum()

(np.float64(0.85), np.float64(6735.0), np.int64(383))

**Explanation:** This code inspects the minimum and maximum item prices for extremes and counts items with zero freight value to verify if free delivery is plausible.

In [19]:
columns = ["product_id", "product_category_name", "product_weight_g",
           "product_length_cm", "product_height_cm", "product_width_cm"]
products.sort_values("product_weight_g", ascending=False)[columns].head(6)

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
25166,26644690fde745fc4654719c3904e1db,cama_mesa_banho,40425.0,13.0,65.0,28.0
11327,8d4e92265a16e69a1e1d76e67e46d72f,instrumentos_musicais,30000.0,80.0,75.0,36.0
31042,5459103ed3b2f852d267e6a70b00dc24,casa_construcao,30000.0,40.0,79.0,40.0
5783,b2b7d701c1f12d5a0f52ec90b0a8f819,beleza_saude,30000.0,55.0,75.0,61.0
27230,e5f2d52b802189ee658865ca93d83a8f,pet_shop,30000.0,50.0,30.0,40.0
27266,ab495f166205a883ffe5ab0b5b55f867,eletrodomesticos,30000.0,62.0,73.0,63.0


**Explanation:** This code sorts products in descending order by weight to identify the heaviest product (40,425 g) in the catalog.

In [20]:
carrier_first = orders["order_delivered_carrier_date"] < orders["order_approved_at"]
delivered_first = orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]

carrier_first.sum(), delivered_first.sum(), (carrier_first | delivered_first).sum()

(np.int64(1359), np.int64(23), np.int64(1382))

**Explanation:** This code tests for chronological anomalies in order timestamps—such as items reaching carriers before approval or customers before carriers—and sums the violations.

In [21]:
orders = orders.assign(dates_ok=~(carrier_first | delivered_first))
orders["dates_ok"].value_counts()

dates_ok
True     98059
False     1382
Name: count, dtype: int64

**Explanation:** This code creates a boolean flag column (dates_ok) to mark orders with valid timelines instead of deleting the real customer records.

In [22]:
KEYS = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "customers": ["customer_id"],
}


def log_rows(table, rows, column, new_value, action, rule):
    key = rows[KEYS[table][0]].astype(str)
    for key_column in KEYS[table][1:]:
        key = key + "/" + rows[key_column].astype(str)
    return pd.DataFrame({
        "table": table,
        "key": key,
        "column": column,
        "old_value": rows[column].astype(str),
        "new_value": new_value,
        "action": action,
        "rule": rule,
    })

**Explanation:** This code defines a reusable log_rows function to track and document every data modification, removal, or flagging action with composite keys and clear reasons.

In [23]:
payments = query("SELECT * FROM order_payments")

no_type = payments["payment_type"] == "not_defined"
zero_installments = payments["payment_installments"] == 0

payments_log = pd.concat([
    log_rows("order_payments", payments[no_type], "payment_type",
             "(row removed)", "removed", "no payment type and no money"),
    log_rows("order_payments", payments[zero_installments], "payment_installments",
             "(missing)", "changed", "0 installments is impossible"),
])
payments_log

,table,key,column,old_value,new_value,action,rule
51280,order_payments,4637ca194b6387e2d538dc89b124b0ee/1,payment_type,not_defined,(row removed),removed,no payment type and no money
57411,order_payments,00b1cb0320190ca0daa2c88b35206009/1,payment_type,not_defined,(row removed),removed,no payment type and no money
94427,order_payments,c8c528189310eaa44a745b8d9d26908b/1,payment_type,not_defined,(row removed),removed,no payment type and no money
46982,order_payments,744bade1fcf9ff3f31d860ace076d422/2,payment_installments,0,(missing),changed,0 installments is impossible
79014,order_payments,1a57108394169c0b47d8f876acc9ba2d/2,payment_installments,0,(missing),changed,0 installments is impossible


**Explanation:** This code isolates invalid payment records (undefined types and zero installments) and compiles a cleaning log dataframe documenting the intended actions.

In [24]:
cleaned_payments = payments[~no_type].astype({"payment_installments": "Int64"})
cleaned_payments = cleaned_payments.assign(
    payment_installments=cleaned_payments["payment_installments"].mask(cleaned_payments["payment_installments"] == 0)
)
len(payments), len(cleaned_payments), cleaned_payments["payment_installments"].isna().sum()

(103886, 103883, np.int64(2))

**Explanation:** This code executes the payment cleaning rules by removing invalid rows, casting installment columns to Int64, and masking zero installments into missing values.

In [25]:
#3.a
zero_weight = products["product_weight_g"] == 0
products_log = log_rows("products", products[zero_weight], "product_weight_g", "(missing)", "changed", "a weight of 0 g is impossible")
products_cleaned = products.assign(product_weight_g=products["product_weight_g"].mask(zero_weight))
zero_weight.sum(), products_cleaned["product_weight_g"].isna().sum()

(np.int64(4), np.int64(6))

**Explanation:** This code identifies products with an impossible weight of 0 grams, logs the changes using log_rows, and masks those weights into missing values.

In [26]:
#3.b
items_orders = items.merge(
    orders[["order_id", "order_purchase_timestamp"]],
    on="order_id",
    validate="many_to_one",
)
late_shipping = items_orders["shipping_limit_date"] > items_orders[
    "order_purchase_timestamp"
] + pd.Timedelta(days=365)
items_log = log_rows(
    "order_items",
    items_orders[late_shipping],
    "shipping_limit_date",
    "(missing)",
    "changed",
    "shipping deadline more than a year after purchase",
)

items_cleaned = items_orders.assign(
    shipping_limit_date=items_orders["shipping_limit_date"].mask(late_shipping)
)
late_shipping.sum(), items_cleaned["shipping_limit_date"].isna().sum()

(np.int64(4), np.int64(4))

**Explanation:** This code merges items with order purchase timestamps to find shipping deadlines set more than a year after purchase, logging and flagging them for correction.

In [27]:
#3.c
purchased_after_carrier = orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]
count_166 = purchased_after_carrier.sum()
not_in_carrier_first = (purchased_after_carrier & ~carrier_first).sum()
count_166, not_in_carrier_first

(np.int64(166), np.int64(0))

**Explanation:** Out of the 166 orders where delivery to carrier occurred before purchase, exactly 0 are outside `carrier_first`, confirming that all chronological timeline violations involving purchase timestamps are fully captured by our date checks.

In [28]:
#3.d
zero_vouchers = payments[(payments["payment_type"] == "voucher") & (payments["payment_value"] == 0)]
len(zero_vouchers)

6

**Explanation:** This code filters for vouchers with a value of 0.00 to evaluate whether they are possible promotional vouchers or invalid data entries.

In [29]:
#3.e
paid_no_items = payments[~payments["order_id"].isin(items["order_id"])]
paid_no_items_orders = orders[orders["order_id"].isin(paid_no_items["order_id"])]

len(paid_no_items_orders), len(paid_no_items), paid_no_items["payment_value"].sum()


(775, 830, np.float64(162591.95))

**Explanation:** This code identifies payments belonging to orders that have no corresponding items in the items table, summarizing their statuses and total payment value.

In [30]:
#3.f
canceled_delivered = orders[(orders["order_status"] == "canceled") & (orders["order_delivered_customer_date"].notna())]
len(canceled_delivered)

6

**Explanation:** This code isolates the 6 canceled orders that still have a recorded customer delivery date to examine potential status tracking discrepancies.

## Part 4 — Text written in different ways


In [31]:
paulo = sellers["seller_city"].str.contains("paulo") | (
    sellers["seller_city"] == "sp"
)
sellers[paulo]["seller_city"].value_counts()

seller_city
sao paulo                     694
sp                              4
sao paulo - sp                  3
paulo lopes                     2
sao paulo / sao paulo           1
são paulo                      1
santo andre/sao paulo           1
maua/sao paulo                  1
sao paulo sp                    1
sao  paulo                      1
ribeirao preto / sao paulo      1
carapicuiba / sao paulo         1
sao paulop                      1
jacarei / sao paulo             1
Name: count, dtype: int64

**Explanation:** This code filters seller cities containing 'paulo' or matching 'sp' to inspect all the different spelling variations present in the raw data.

In [32]:
def normalize_city(cities):
  return (
      cities.str.strip()
      .str.lower()
      .str.normalize("NFKD")
      .str.encode("ascii", errors="ignore")
      .str.decode("ascii")
      .str.replace("'", " ")
      .str.replace(r"\s+", " ", regex=True)
  )

**Explanation:** This code defines a normalization function that strips whitespace, lowercases text, removes accent marks, standardizes apostrophes, and collapses multiple spaces.

In [33]:
city = normalize_city(sellers["seller_city"])
city = city.str.split(r" - |[/\\,]", regex=True).str[0].str.strip()

**Explanation:** This code applies city normalization and splits text at separators like hyphens, slashes, or commas, keeping only the primary city name part.

In [34]:
state = sellers["seller_state"].str.lower()
ends_with_state = (city.str[-3:] == " " + state) | (
    city.str[-3:] == "-" + state
)
city = city.mask(ends_with_state, city.str[:-3])

**Explanation:** This code checks if the last three characters of the city name match the state suffix and trims them off to ensure cleaner city designations.

In [35]:
CITY_FIXES = {
    "sao paulop": "sao paulo",
    "sbc": "sao bernardo do campo",
}
city = city.replace(CITY_FIXES)

**Explanation:** This code uses a dictionary replacement mapping to fix specific known typos and abbreviations in the city names.

In [36]:
not_a_city = city.str.contains(r"[0-9@]", regex=True) | (city == state)
sellers[not_a_city]["seller_city"].tolist()

['04482255', 'sp / sp', 'sp', 'sp', 'sp', 'vendas@creditparts.com.br', 'sp']

**Explanation:** This code flags entries that contain digits, symbols (@), or consist solely of state abbreviations, confirming they are not valid city names.

In [37]:
city = city.mask(not_a_city)
cleaned_sellers = sellers.assign(seller_city=city)

cleaned_sellers.groupby(["seller_city", "seller_state"]).size().sort_values(
    ascending=False
).head(6)

seller_city     seller_state
sao paulo       SP              702
curitiba        PR              124
rio de janeiro  RJ               96
belo horizonte  MG               66
ribeirao preto  SP               53
guarulhos       SP               50
dtype: int64

**Explanation:** This code masks non-city entries into missing values, updates the sellers dataframe, and displays the top cities by seller count after cleaning.

In [38]:
sellers["seller_city"].nunique(), cleaned_sellers["seller_city"].nunique()

(611, 578)

**Explanation:** This code compares the count of unique city spellings before and after cleaning to demonstrate the reduction in text inconsistencies.

In [39]:
#4.a
geo_cities = query("SELECT geolocation_city FROM geolocation")
geo_before = geo_cities["geolocation_city"].nunique()
geo_normalized = normalize_city(geo_cities["geolocation_city"])
geo_after = geo_normalized.nunique()
geo_before, geo_after

(8011, 5942)

**Explanation:** This code runs the normalization function on the geolocation table cities to compare unique spelling counts before and after text cleaning.

In [40]:
#4.b
sbc_sellers = sellers[sellers["seller_city"].isin(["sbc", "sbc/sp"])]
sbc_zip = sbc_sellers["seller_zip_code_prefix"].tolist()
customers = query("SELECT customer_zip_code_prefix, customer_city FROM customers")
sbc_zip, customers[customers["customer_zip_code_prefix"].isin(sbc_zip)]["customer_city"].value_counts().head(3)

(['09726', '09861'],
 customer_city
 sao bernardo do campo    46
 Name: count, dtype: int64)

**Explanation:** This code finds the zip code prefixes for sellers using the abbreviation 'sbc' and checks customer cities with the same prefix to prove it stands for São Bernardo do Campo.